# トルバプタン在宅導入研究：採血データ安全性解析

159人の導入患者について、Na、K、Cre、BUNの採血実施率、検査値分布、Baselineからの変化量、Na/K異常値を集計するノートブックです。

## 解析定義
- **Baseline**：導入前30日以内〜導入当日（day 0）。期間内で導入日に最も近い採血を患者・検査項目ごとに採用します。day 0は内服前として扱います。
- **追跡期間**：1–3日、4–7日、8–30日、31日以降。
- 各追跡期間では、患者・検査項目ごとに期間内で最も早い採血を採用します。
- 実施率の分母は、データ内の全対象患者数です。
- Na異常：135未満／145超、K異常：3.5未満／5.0超。
- Cre上昇候補は全採血記録から探索します。ただし、採血日だけではKDIGOの48時間基準を厳密に判定できず、候補は確定AKIではありません。

**重要：** 元CSVに検査単位・採血時刻がない場合、それらを自動推定しません。臨床発表前に単位を確認してください。

## 1. 入力ファイルとライブラリ

CSVの場所を `CSV_PATH` に指定してください。初期値は、このノートブックと同じフォルダにCSVがある場合のファイル名です。必要に応じて絶対パスに変更してください。

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# 必要に応じて絶対パスに変更
CSV_PATH = Path("tolvaptan_lab_long_HF1.csv")
OUTPUT_XLSX = Path("tolvaptan_safety_summary.xlsx")

# 検査項目コード（元データの item_code）
ITEM_CODES = {
    "Na": 87,
    "K": 93,
    "Cre": 74,
    "BUN": 63,
}

# 異常値の閾値（単位は施設の検査システムで確認すること）
NA_LOW = 135
NA_HIGH = 145
K_LOW = 3.5
K_HIGH = 5.0

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"CSVが見つかりません: {CSV_PATH.resolve()}\n"
        "CSV_PATHを実際のファイルパスに変更してください。"
    )

df = pd.read_csv(CSV_PATH)
print(f"行数: {len(df):,}")
print(f"患者数: {df['Patient_ID'].nunique():,}")
display(df.head())


## 2. データ検証と日数の再計算

`days_diff` は `sample_date - ind_date` から再計算します。日付の欠損・不正値、必須列の欠落を確認します。

In [ ]:
REQUIRED_COLUMNS = {
    "Patient_ID", "item_code", "item_name", "ind_date",
    "sample_date", "value_raw", "value_num"
}
missing_columns = REQUIRED_COLUMNS - set(df.columns)
if missing_columns:
    raise ValueError(f"必須列がありません: {sorted(missing_columns)}")

df["ind_date"] = pd.to_datetime(df["ind_date"], errors="coerce")
df["sample_date"] = pd.to_datetime(df["sample_date"], errors="coerce")
df["item_code"] = pd.to_numeric(df["item_code"], errors="coerce").astype("Int64")
df["value_num"] = pd.to_numeric(df["value_num"], errors="coerce")
df["days_diff"] = (df["sample_date"] - df["ind_date"]).dt.days

print("日付欠損/変換不能:", int(df["ind_date"].isna().sum()), "ind_date;",
      int(df["sample_date"].isna().sum()), "sample_date")
print("数値化できない検査値:", int(df["value_num"].isna().sum()))
print("患者数:", df["Patient_ID"].nunique())
print("検査項目コード:", sorted(df["item_code"].dropna().astype(int).unique()))


## 3. Baselineと追跡期間の代表値を選択

同一患者・同一検査項目・同一期間に複数の採血がある場合、Baselineは導入日に最も近い採血、追跡期間は最も早い採血を選択します。追跡期間の境界は重複しません。

In [ ]:
PERIODS = [
    ("Baseline", -30, 0),
    ("1–3日", 1, 3),
    ("4–7日", 4, 7),
    ("8–30日", 8, 30),
    ("31日以降", 31, np.inf),
]

N_PATIENTS = df["Patient_ID"].nunique()

def select_period_records(data: pd.DataFrame, label: str, low: int, high: float) -> pd.DataFrame:
    subset = data[data["days_diff"].between(low, high, inclusive="both")].copy()
    if label == "Baseline":
        # day 0を含め、0に最も近い値（day_diffが最大）を採用
        subset = subset.sort_values(
            ["Patient_ID", "item_code", "days_diff", "sample_date"],
            ascending=[True, True, False, False]
        )
    else:
        # 期間内で最も早い採血を採用
        subset = subset.sort_values(
            ["Patient_ID", "item_code", "days_diff", "sample_date"],
            ascending=[True, True, True, True]
        )
    return subset.drop_duplicates(["Patient_ID", "item_code"], keep="first")

selected = {
    label: select_period_records(df, label, low, high)
    for label, low, high in PERIODS
}

for label, _, _ in PERIODS:
    print(label, "records:", len(selected[label]))


## 4. 採血実施率と検査値分布

各期間について、測定患者数、全対象患者を分母とした実施率、中央値、四分位範囲、最小値・最大値を計算します。Na/Kは異常値数と、当該期間に測定された患者を分母とする割合も計算します。

In [ ]:
def format_n_pct(n, denominator):
    if denominator == 0:
        return f"{n} (NA)"
    return f"{n} ({100*n/denominator:.1f}%)"

summary_rows = []

for analyte, code in ITEM_CODES.items():
    for period, _, _ in PERIODS:
        sub = selected[period]
        sub = sub[sub["item_code"] == code].dropna(subset=["value_num"])
        values = sub["value_num"]
        n_measured = int(sub["Patient_ID"].nunique())

        row = {
            "検査項目": analyte,
            "時点": period,
            "測定患者数": n_measured,
            "実施率（分母全患者、%）": round(100 * n_measured / N_PATIENTS, 1)
                if N_PATIENTS else np.nan,
            "中央値": values.median(),
            "第1四分位": values.quantile(0.25),
            "第3四分位": values.quantile(0.75),
            "最小値": values.min(),
            "最大値": values.max(),
        }

        if analyte == "Na":
            n_low = int((values < NA_LOW).sum())
            n_high = int((values > NA_HIGH).sum())
            row["低値 n（測定者中%）"] = format_n_pct(n_low, n_measured)
            row["高値 n（測定者中%）"] = format_n_pct(n_high, n_measured)
        elif analyte == "K":
            n_low = int((values < K_LOW).sum())
            n_high = int((values > K_HIGH).sum())
            row["低値 n（測定者中%）"] = format_n_pct(n_low, n_measured)
            row["高値 n（測定者中%）"] = format_n_pct(n_high, n_measured)

        summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
display(summary_df.round(2))


## 5. Baselineからの変化量（同一患者内の比較）

各追跡期間の値とBaseline値が両方ある患者のみを対象に、追跡値−Baselineの変化量を計算します。期間ごとに比較可能な患者数が異なります。

In [ ]:
change_rows = []

for analyte, code in ITEM_CODES.items():
    base = selected["Baseline"]
    base = base[base["item_code"] == code][["Patient_ID", "value_num"]].rename(
        columns={"value_num": "Baseline"}
    )

    for period, _, _ in PERIODS:
        if period == "Baseline":
            continue
        follow = selected[period]
        follow = follow[follow["item_code"] == code][["Patient_ID", "value_num"]].rename(
            columns={"value_num": "Followup"}
        )

        paired = base.merge(follow, on="Patient_ID", how="inner").dropna(
            subset=["Baseline", "Followup"]
        )
        delta = paired["Followup"] - paired["Baseline"]

        change_rows.append({
            "検査項目": analyte,
            "時点": period,
            "ペア数": int(len(paired)),
            "変化量中央値": delta.median(),
            "変化量第1四分位": delta.quantile(0.25),
            "変化量第3四分位": delta.quantile(0.75),
            "変化量平均（参考）": delta.mean(),
        })

changes_df = pd.DataFrame(change_rows)
display(changes_df.round(3))


## 6. Baselineからの異常値の新規発生

Na/Kについて、Baselineでは閾値異常がなく、追跡時点で閾値異常となった患者を数えます。これは「その時点の値」で判定した記述的集計であり、期間内の全採血における発生を完全に捉えるものではありません。

In [ ]:
incident_rows = []

for analyte, code, low_thr, high_thr in [
    ("Na", ITEM_CODES["Na"], NA_LOW, NA_HIGH),
    ("K", ITEM_CODES["K"], K_LOW, K_HIGH),
]:
    base = selected["Baseline"]
    base = base[base["item_code"] == code][["Patient_ID", "value_num"]].rename(
        columns={"value_num": "Baseline"}
    )

    def abnormal(s):
        return (s < low_thr) | (s > high_thr)

    for period, _, _ in PERIODS:
        if period == "Baseline":
            continue
        follow = selected[period]
        follow = follow[follow["item_code"] == code][["Patient_ID", "value_num"]].rename(
            columns={"value_num": "Followup"}
        )
        paired = base.merge(follow, on="Patient_ID", how="inner").dropna()
        baseline_normal = ~abnormal(paired["Baseline"])
        follow_abnormal = abnormal(paired["Followup"])
        new_abnormal = baseline_normal & follow_abnormal
        denominator = int(baseline_normal.sum())

        incident_rows.append({
            "検査項目": analyte,
            "時点": period,
            "Baseline正常かつ追跡値あり": denominator,
            "Baseline正常から異常化 n": int(new_abnormal.sum()),
            "Baseline正常から異常化（%）": (
                round(100 * new_abnormal.sum() / denominator, 1)
                if denominator else np.nan
            ),
        })

incident_df = pd.DataFrame(incident_rows)
display(incident_df)


## 7. Cre上昇の探索的スクリーニング

次の条件に合う連続採血ペアを抽出します。
- Creが0.3 mg/dL以上上昇し、採血日の日付差が1日以内
- Creが前値の1.5倍以上となり、採血日の日付差が7日以内

**制限：** 採血時刻がないため、日付差1日は48時間以内を厳密に意味しません。また、KDIGOのAKI診断には臨床状況や尿量なども関係します。出力はカルテ確認用の候補であり、AKI確定例数・発生率ではありません。

In [ ]:
cre = df[df["item_code"] == ITEM_CODES["Cre"]].dropna(
    subset=["Patient_ID", "sample_date", "value_num", "days_diff"]
).copy()

aki_candidate_rows = []

for patient_id, group in cre.groupby("Patient_ID"):
    group = group.sort_values("sample_date").drop_duplicates("sample_date", keep="first")
    records = group.to_dict("records")

    for i, previous in enumerate(records):
        for current in records[i + 1:]:
            gap_days = (current["sample_date"] - previous["sample_date"]).days
            # 導入後30日以内に起きた上昇を候補として抽出
            if gap_days <= 0 or current["days_diff"] < 1 or current["days_diff"] > 30:
                continue

            rise = current["value_num"] - previous["value_num"]
            ratio = current["value_num"] / previous["value_num"] if previous["value_num"] > 0 else np.nan

            if gap_days <= 1 and rise >= 0.3:
                aki_candidate_rows.append({
                    "Patient_ID": patient_id,
                    "候補基準": "Cre上昇≥0.3 mg/dL、採血日差≤1日（48時間基準の近似）",
                    "前回採血日": previous["sample_date"],
                    "今回採血日": current["sample_date"],
                    "前値days_diff": previous["days_diff"],
                    "今回days_diff": current["days_diff"],
                    "日付差（日）": gap_days,
                    "前値Cre": previous["value_num"],
                    "今回Cre": current["value_num"],
                    "上昇量": rise,
                    "倍率": ratio,
                })

            if gap_days <= 7 and previous["value_num"] > 0 and ratio >= 1.5:
                aki_candidate_rows.append({
                    "Patient_ID": patient_id,
                    "候補基準": "Cre≥1.5倍、採血日差≤7日",
                    "前回採血日": previous["sample_date"],
                    "今回採血日": current["sample_date"],
                    "前値days_diff": previous["days_diff"],
                    "今回days_diff": current["days_diff"],
                    "日付差（日）": gap_days,
                    "前値Cre": previous["value_num"],
                    "今回Cre": current["value_num"],
                    "上昇量": rise,
                    "倍率": ratio,
                })

aki_candidates_df = pd.DataFrame(aki_candidate_rows)
if aki_candidates_df.empty:
    print("該当するCre上昇候補ペアはありませんでした。")
else:
    display(aki_candidates_df.sort_values(["Patient_ID", "今回採血日"]))
    print("候補ペア数:", len(aki_candidates_df))
    print("候補患者数:", aki_candidates_df["Patient_ID"].nunique())


## 8. Excelに出力

結果表をExcelの別シートに出力します。元データの検査単位が確認できたら、発表資料に単位を明記してください。

In [ ]:
metadata_df = pd.DataFrame([
    {"項目": "患者数", "内容": N_PATIENTS},
    {"項目": "元CSVレコード数", "内容": len(df)},
    {"項目": "Baseline", "内容": "導入前30日以内〜導入当日（day 0を内服前として含める）"},
    {"項目": "追跡期間", "内容": "1–3日、4–7日、8–30日、31日以降"},
    {"項目": "各期間の代表値", "内容": "患者・検査項目ごとに期間内で最も早い採血"},
    {"項目": "Na異常閾値", "内容": f"<{NA_LOW} または >{NA_HIGH}（単位は検査システムで確認）"},
    {"項目": "K異常閾値", "内容": f"<{K_LOW} または >{K_HIGH}（単位は検査システムで確認）"},
    {"項目": "Cre候補判定", "内容": "採血日ベースの探索。確定AKI診断や発生率ではない"},
    {"項目": "制限", "内容": "採血時刻、検査単位、臨床症状・対応がCSVにない場合は別途確認が必要"},
])

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="検査値と実施率", index=False)
    changes_df.to_excel(writer, sheet_name="Baselineからの変化", index=False)
    incident_df.to_excel(writer, sheet_name="NaK新規異常", index=False)
    aki_candidates_df.to_excel(writer, sheet_name="Cre上昇候補", index=False)
    metadata_df.to_excel(writer, sheet_name="定義と注意点", index=False)

    for ws in writer.book.worksheets:
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions
        for col in ws.columns:
            max_len = max(
                len(str(cell.value)) if cell.value is not None else 0
                for cell in col
            )
            ws.column_dimensions[col[0].column_letter].width = min(max(max_len + 2, 12), 58)

print(f"Excel出力完了: {OUTPUT_XLSX.resolve()}")
